**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/andreassen-2022"], check=True)
    os.chdir("/content/encore/entries/andreassen-2022")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Andreassen et al. (2022) from FIGSHARE:19640805

**Paper.** Andreassen AH, Hall P, Khatibzadeh P, Jutfelt F, Kermen F (2022). Brain dysfunction
during warming is linked to oxygen limitation in larval zebrafish. *PNAS* 119, e2207052119.
[doi:10.1073/pnas.2207052119](https://doi.org/10.1073/pnas.2207052119). The claims file
(`claims/andreassen-2022.yaml`) was written from the version of record (PMC9522358) and its SI
Appendix, with Tables S1 to S10 read from the rendered PDF.

**Dataset.** [FIGSHARE:19640805](https://doi.org/10.6084/m9.figshare.19640805.v4), "Data and R
script for: Brain dysfunction during warming is linked to oxygen limitation in larval
zebrafish", version 4 (doi:10.6084/m9.figshare.19640805.v4, published 2022-09-02). The paper
cites the item without a version (Data Availability, doi:10.6084/m9.figshare.19640805). Version 4
is the most recent published version and is pinned for that reason. Versions 1 and 2 hold only
the R script and the Readme; version 3 holds the same twelve files as version 4, with the same
MD5s (printed in the data check), so the choice between 3 and 4 changes nothing.

The item holds ten CSV tables of derived per-fish values, the Readme.txt that describes their
columns, and the paper's R script, `Code_Andreassen_et_al_2022.R`. There is no imaging and no
time series. The MD5 of each file read is printed in the first data-check cell and checked
against the archive's `computed_md5` as the file is streamed.

**Method.** The models, reference levels, outlier rule and variable transformations are those of
the paper's R script, which the claims file names as the place where open details may be
settled; each section cites the lines it follows. Mixed models are fitted by REML with a random
intercept per fish (`thermal.LMM`), and the binomial model by maximum likelihood with the Laplace
approximation (`thermal.BinomialGLMM`), in Python, since R is not part of the locked environment.
No analysis uses random draws, so no seeds are needed.

| Claim | Tested | Data |
|---|---|---|
| fig1c-ctmax-5dpf | yes | CTmax5and9dpfAdults.csv |
| fig1c-ctmax-similar-across-ages | yes | CTmax5and9dpfAdults.csv |
| figs1a-swim-speed-no-treatment-difference | yes | Swimspeed.csv |
| figs1bc-spiral-swimming | blocked, data missing | no table; the R script types the counts in |
| figs1d-loss-of-equilibrium | yes | LOEData.csv |
| figs1e-loss-of-response-specific | blocked, data missing | no loss-of-response record for control fish |
| fig2f-medulla-control-stable | yes | CalciumPeaksLong.csv |
| fig2f-medulla-heat-rise-then-fall | yes | CalciumPeaksLong.csv |
| figs2a-medulla-decrease-39-41 | blocked, data missing | CalciumPeaksLong.csv has no temperature |
| fig2g-depolarization-onset-temperature | blocked, data missing | experiment 2 onsets not deposited |
| fig2e-depolarization-spread | blocked, data missing | no imaging |
| fig2d-recovery-after-depolarization | blocked, data missing | no time series after the depolarization |
| fig3d-ctmax-precedes-depolarization | yes | CTmaxandDepolarization.csv |
| fig4c-heart-rate-rises-to-34 | yes | Heartbeat.csv |
| fig4c-heart-rate-falls-then-stable | yes | Heartbeat.csv |
| figs2b-medulla-lateral-mount | blocked, data missing | experiment 4 medulla counts not deposited |
| fig5a-ctmax-oxygen | yes | CTmaxData_Total.csv |
| fig5b-depolarization-oxygen | yes | DepolarizationPeakandRecovery.csv |
| figs3b-amplitude-no-oxygen-effect | yes | DepolarizationPeakandRecovery.csv |
| figs3c-recovery-longer-in-hypoxia | yes | DepolarizationPeakandRecovery.csv |
| figs3d-medulla-drops-before-depolarization | yes, on 1 °C bins | CalciumPeakExperiment.csv, DepolarizationPeakandRecovery.csv |
| fig5d-medulla-normoxia | yes | CalciumPeakExperiment.csv |
| fig5d-medulla-hyperoxia | yes | CalciumPeakExperiment.csv |
| fig5d-medulla-hypoxia | yes | CalciumPeakExperiment.csv |
| figs4ab-light-response-amplitude-falls | blocked, data missing | only response counts, no amplitudes |
| fig6c-light-responsiveness-by-oxygen | yes | LightResponse.csv |
| tables10-light-response-model | yes | LightResponse.csv |
| discussion-depolarization-magnitude | blocked, data missing | no medulla peak amplitudes |

In [ ]:
import re
import time
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

import thermal

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
files, raw = thermal.load_all()
meta = files.attrs["meta"]
print(f"FIGSHARE:{thermal.ARTICLE_ID} version {meta['version']}, doi {meta['doi']}, "
      f"published {meta['published_date']}")
print(f"streamed {len(raw)} files, {sum(map(len, raw.values()))} bytes, in {time.time() - started:.1f} s")

tables = {name: thermal.read_csv(raw, name) for name in sorted(raw) if name.endswith(".csv")}
script = raw["Code_Andreassen_et_al_2022.R"].decode("latin-1").replace("\r\n", "\n")
readme = raw["Readme.txt"].decode("latin-1").replace("\r", "\n")

## Data check

In [ ]:
print("Files of version 4 (MD5 checked against computed_md5 when streamed):")
print(files[["id", "name", "size", "computed_md5"]].to_string(index=False))

import json, urllib.request
with urllib.request.urlopen(f"https://api.figshare.com/v2/articles/{thermal.ARTICLE_ID}/versions") as r:
    versions = json.load(r)
print("\nPublished versions:", [v["version"] for v in versions])
with urllib.request.urlopen(next(v["url"] for v in versions if v["version"] == 3)) as r:
    v3 = json.load(r)
same = {f["name"]: f["computed_md5"] for f in v3["files"]} == dict(zip(files.name, files.computed_md5))
print("Version 3 has the same file names and MD5s as version 4:", same)

In [ ]:
# Every column of every table, with its type, count of values and range.
for name, table in tables.items():
    print(f"== {name}: {table.shape[0]} rows")
    for column in table.columns:
        values = table[column]
        if pd.api.types.is_numeric_dtype(values):
            desc = f"numeric, {values.notna().sum()} values, {values.min()} to {values.max()}"
        else:
            levels = values.dropna().unique()
            desc = f"text, {values.notna().sum()} values, {len(levels)} levels: {', '.join(map(str, levels[:8]))}" + (" ..." if len(levels) > 8 else "")
        print(f"   {column}: {desc}")

The Readme is printed next, and each table's columns are checked against the section that
describes it. LightResponse.csv has a row-name field at the start of each row that the Readme does
not describe (dropped on reading; the header has one name fewer than the rows). The Readme gives
`Recovery` of DepolarizationPeakandRecovery.csv in minutes, but its values (60 to 464) are
seconds, which the R script divides by 60 (`RecoveryMin = round(Recovery/60, 2)`); this is shown in
the cell after. The `Response` column of LOEData.csv is, by the Readme, whether the fish ever lost
equilibrium (0 or 1), not loss of response. The counts of subjects per experiment are checked after
that against the paper.

In [ ]:
print(readme)
# Each table's columns against the Readme section that describes it.
sections = re.split(r"\n(?=\S+\.(?:csv|R)\s*\n#-#)", readme)
for name, table in tables.items():
    section = next(s for s in sections if s.startswith(name))
    described = [c for c in table.columns if re.search(r"(?m)^" + re.escape(c) + r"\s", section)]
    print(f"{name}: {len(described)} of {len(table.columns)} columns described;",
          "not described:", [c for c in table.columns if c not in described])

In [ ]:
# finding-recovery-unit: the Readme gives Recovery in minutes; the values are seconds
recovery = tables["DepolarizationPeakandRecovery.csv"].Recovery
print("Readme:", [" ".join(l.split()) for l in readme.split("\n") if l.startswith("Recovery")])
print(f"Recovery: {recovery.count()} values, {recovery.min()} to {recovery.max()}, median {recovery.median()}")
print("R script:", [l.strip() for l in script.split("\n") if "RecoveryMin=" in l.replace(" ", "")][:1])

In [ ]:
ages = tables["CTmax5and9dpfAdults.csv"]
swim = tables["Swimspeed.csv"]
loe = tables["LOEData.csv"]
peaks2 = tables["CalciumPeaksLong.csv"]
exp3 = tables["CTmaxandDepolarization.csv"]
heart = tables["Heartbeat.csv"]
ct_o2 = tables["CTmaxData_Total.csv"]
depol = tables["DepolarizationPeakandRecovery.csv"]
peaks5 = tables["CalciumPeakExperiment.csv"]
light = tables["LightResponse.csv"]

print("Experiment 1, CTmax by age (paper: 12 at 5 dpf, 12 at 9 dpf, 16 adults):")
print(ages.groupby(["AgeGroup", "Age"]).CTmax.agg(["count", "min", "max"]).to_string())
print("\nExperiment 1, swim speed (paper: 14 control, 12 heat ramp, 5 intervals, 130 observations):")
print(swim.groupby("Treatment").agg(fish=("FishID", "nunique"), rows=("SwimSpeed", "size"),
                                    min=("SwimSpeed", "min"), max=("SwimSpeed", "max")).to_string())
print("Temperature labels by treatment:", swim.groupby("Treatment").Temperature.unique().to_dict())
print("\nExperiment 1, loss of equilibrium (paper: 14 control, 12 heat ramp):")
print(loe.groupby("Treatment").LOE.agg(["count", "min", "max"]).to_string())
print("\nExperiment 2, medulla peaks (paper: 8 control, 11 heat ramp, 266 observations):")
print(peaks2.groupby("Treatment").agg(fish=("FishID", "nunique"), rows=("Interval", "size"),
                                      intervals=("Interval", lambda s: sorted(s.unique()))).to_string())
print("Frequency3min range:", peaks2.Frequency3min.min(), "to", peaks2.Frequency3min.max(), "peaks per 3 min")
print("\nExperiment 3 (paper: 7 fish, 14 observations):", exp3.FishID.nunique(), "fish,", len(exp3), "rows,",
      exp3.Response.value_counts().to_dict(), "range", exp3.Temperature.min(), "to", exp3.Temperature.max(), "°C")
print("\nExperiment 4, heart rate (paper: 9 fish, 50 observations):", heart.FishID.nunique(), "fish,",
      heart.HeartBeat.notna().sum(), "values of", len(heart), "rows; categories",
      list(heart.Temperature.unique()), "; range", heart.HeartBeat.min(), "to", heart.HeartBeat.max(), "Hz")
print("\nExperiment 5, CTmax (paper: 76 fish with a value before outliers):")
print(ct_o2.groupby(["Treatment", "Experiment"], dropna=False).Temperature.agg(["size", "count", "min", "max"]).to_string())
print("\nExperiment 5, depolarization (paper: 14 hypoxia, 12 normoxia, 14 hyperoxia):")
print(depol.groupby(["Treatment", "Experiment"]).agg(fish=("FishID", "size"), onset_min=("DepolTemp", "min"),
      onset_max=("DepolTemp", "max"), peak_min=("Peak", "min"), peak_max=("Peak", "max"),
      recovery_s_min=("Recovery", "min"), recovery_s_max=("Recovery", "max"), recovery_n=("Recovery", "count")).to_string())
print("\nExperiment 5, medulla frequency (paper: 12 hypoxia, 10 normoxia, 11 hyperoxia; 391 observations):")
print(peaks5.groupby("Treatment").agg(fish=("FishID", "nunique"), values=("Frequency", "count"),
      lower_min=("LowerTemp", "min"), lower_max=("LowerTemp", "max"), f_max=("Frequency", "max")).to_string())
print("AvgTemp - LowerTemp:", (peaks5.AvgTemp - peaks5.LowerTemp).unique(),
      "; values from LowerTemp 28 on:", peaks5[peaks5.LowerTemp > 27].Frequency.notna().sum())
print("\nExperiment 5, light responses (paper: 14 hypoxia, 11 normoxia, 13 hyperoxia; 38 fish, 2763 flashes):")
lr = light[light.Temperature > 28]
print(light.groupby("Treatment").FishID.nunique().to_string())
print("flashes with an outcome, Temperature above 28:", int((lr["number.of.successes"] + lr["number.of.failures"]).sum()))
print("denominator equals successes + failures where both are given:",
      bool(((light["number.of.successes"] + light["number.of.failures"]) == light["binomial.denominator"])[light.prob.notna()].all()),
      "; rows without counts:", int(light.prob.isna().sum()))
print("prob equals successes / denominator:",
      bool(np.allclose((light["number.of.successes"] / light["binomial.denominator"])[light.prob.notna()], light.prob.dropna(), atol=6e-4)))

Every count matches the paper: 12, 12 and 16 fish for Fig. 1C; 14 control and 12 heat-ramp fish
with 130 speed values; 8 and 11 fish with 266 medulla values (14 three-minute intervals each);
7 fish in experiment 3; 9 fish with 50 heart-rate values; 28, 25 and 23 CTmax values in normoxia,
hypoxia and hyperoxia (76); 12, 14 and 14 depolarizations with 39 recovery times; 10, 12 and 11
fish with 391 medulla values from 28 °C on; 11, 14 and 13 fish with 2763 flashes. Temperatures
are in °C (27 to 43), frequencies in peaks per 3 min (experiment 2) or per minute (experiment 5),
heart rate in Hz (2.2 to 6.4), amplitude in %ΔF/F0 (17 to 206) and recovery in seconds (60 to 464;
the R script divides by 60 for minutes). In LightResponse.csv `binomial.denominator` counts
flashes including those without an outcome in rows where the counts are missing; the model uses
successes and failures, as the R script does.

The tables are derived values; there are no streams with clocks to compare. What can be checked
is whether the two tables of embedded experiment-5 fish describe the same recordings: the
medulla frequencies are in 1 °C bins that should end at the depolarization onset.

In [ ]:
# Are the medulla bins and the light-response bins of each embedded fish of experiment 5 consistent
# with that fish's depolarization onset temperature? The Readme says FishID corresponds across
# these three tables.
last_bin = peaks5.dropna(subset=["Frequency"]).groupby("FishID").LowerTemp.max()
last_light = light.dropna(subset=["prob"]).groupby("FishID").Temperature.max()   # upper bound of its bin
link = depol.set_index("FishID")[["Treatment", "DepolTemp"]].join(last_bin.rename("last_medulla_bin"), how="outer") \
    .join(last_light.rename("last_light_bin_upper"), how="outer")
link["medulla_ends_in_onset_bin"] = link.last_medulla_bin == np.floor(link.DepolTemp)
print(link.to_string())
both = link.dropna(subset=["DepolTemp", "last_medulla_bin"])
print(f"\nFish in the medulla table and the depolarization table: {len(both)}; "
      f"last medulla bin is the 1 °C bin holding the onset in {int(both.medulla_ends_in_onset_bin.sum())}")
print("In the medulla table, not in the depolarization table:", sorted(set(last_bin.index) - set(depol.FishID)))
print("In the light table, not in the depolarization table:", sorted(set(last_light.index) - set(depol.FishID)))
padded = {f: re.sub(r"Fish(\d)$", r"Fish0\1", f) for f in last_light.index}
print("Light-table IDs that match a depolarization ID once zero-padded:",
      {f: g for f, g in padded.items() if f != g and g in set(depol.FishID)})
print("Light recordings ending 5 °C or more below the onset:",
      link[(link.DepolTemp - link.last_light_bin_upper) >= 5][["DepolTemp", "last_light_bin_upper"]].to_dict("index"))

Of the 32 fish in both the medulla and the depolarization tables, 30 have their last medulla bin
in the 1 °C bin that holds their onset temperature, which is consistent with the tables
describing the same recordings and with the medulla series ending at the depolarization. The
exceptions are Fish03 (normoxia; last bin 39 to 40 °C, onset 40.4 °C) and Fish35 (normoxia; last bin
39 to 40 °C, onset 41.1 °C). Fish33 (hypoxia) has medulla and light values but no depolarization
row; the hypoxia count of Table S8 (14) holds without it, and the 12 hypoxic fish of Fig. 5D include it. LightResponse.csv
names three fish Fish3, Fish4 and Fish5, which match the depolarization table's Fish03, Fish04 and
Fish05 only once zero-padded, although the Readme says the IDs correspond across the three
tables. The light recordings of Fish27 and Fish29 end about 10 °C below their onsets (at 30 to 31 °C and
28 to 29 °C), which the paper does not mention; they only contribute fewer flashes to the light
analyses.
The next cell shows these, with the row counts of CTmaxData_Total.csv.

In [ ]:
# finding-fish-ids: medulla series that end before the onset bin, an ID without a depolarization row,
# and IDs spelled differently across tables that the Readme says correspond
print(depol.loc[depol.FishID.isin(["Fish03", "Fish35"]), ["FishID", "Treatment", "DepolTemp"]].to_string(index=False))
for f in ["Fish03", "Fish35"]:
    print(f, "last medulla bins:", peaks5[peaks5.FishID == f].dropna(subset=["Frequency"]).LowerTemp.tail(2).tolist())
print("Fish33 rows in DepolarizationPeakandRecovery.csv:", int((depol.FishID == "Fish33").sum()),
      "; in CalciumPeakExperiment.csv with values:", int(peaks5[peaks5.FishID == "Fish33"].Frequency.notna().sum()))
print("FishIDs in LightResponse.csv absent from DepolarizationPeakandRecovery.csv:",
      sorted(set(light.FishID) - set(depol.FishID)), "; Fish03/Fish04/Fish05 in the depolarization table:",
      sorted(set(depol.FishID) & {"Fish03", "Fish04", "Fish05"}))
for f in ["Fish27", "Fish29"]:
    print(f, "onset", depol.loc[depol.FishID == f, "DepolTemp"].item(), "°C; last light bin (upper bound):",
          light[light.FishID == f].dropna(subset=["prob"]).Temperature.max(), "°C")
print("CTmaxData_Total.csv rows:", len(ct_o2), "; without a treatment:", int(ct_o2.Treatment.isna().sum()),
      "; with a treatment but no temperature:", int((ct_o2.Treatment.notna() & ct_o2.Temperature.isna()).sum()))

In [ ]:
# Recordings deposited twice: identical per-fish series within each table.
def duplicate_series(table, key, value, by):
    series = table.pivot_table(index=key, columns=by, values=value, aggfunc="first")
    pairs = []
    ids = list(series.index)
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            a, b = series.loc[ids[i]], series.loc[ids[j]]
            common = a.notna() & b.notna()
            if common.sum() >= 3 and np.allclose(a[common], b[common]):
                pairs.append((ids[i], ids[j], int(common.sum())))
    return pairs

print("CalciumPeaksLong, identical series:", duplicate_series(peaks2, "FishID", "Frequency3min", "Interval"))
print("CalciumPeakExperiment, identical series:", duplicate_series(peaks5, "FishID", "Frequency", "LowerTemp"))
print("Heartbeat, identical series:", duplicate_series(heart, "FishID", "HeartBeat", "Temperature"))
print("Swimspeed, identical series:", duplicate_series(swim, "FishID", "SwimSpeed", "Time"))
print("LightResponse, identical series:", duplicate_series(light, "FishID", "prob", "Temperature"))
dup_rows = depol.duplicated(subset=["DepolFrame", "DepolTemp", "Peak", "Recovery"]).sum()
print("DepolarizationPeakandRecovery, duplicated rows of the measures:", int(dup_rows))
print("Experiment 3 fish IDs:", sorted(exp3.FishID.unique()))

No per-fish series is repeated within a table, and no depolarization row is repeated, so no
recording appears to be deposited twice. The FishIDs of experiments 1 to 4 restart in each table
(the Readme says so), so fish cannot be followed across those tables; none of the claims needs
that.

The data the claims file describes but that are not in the item are listed in the next cell.

In [ ]:
# finding-missing-measures: measures the paper reports that the item does not hold
print("Columns of CalciumPeaksLong.csv (experiment 2):", list(peaks2.columns), "- no temperature, no onset temperature")
print("Treatments in DepolarizationPeakandRecovery.csv:", sorted(depol.Treatment.unique()),
      "- the only onset temperatures, all from experiment 5 (n =", len(depol), ")")
print("Columns of Heartbeat.csv (experiment 4):", list(heart.columns), "- no medulla counts")
print("Columns of LightResponse.csv:", list(light.columns), "- counts of responses only, no amplitudes")
print("Columns of LOEData.csv:", list(loe.columns), "- loss of equilibrium; no spiral swimming or loss of response")
print("Readme on LOEData Response:", [" ".join(l.split()) for l in next(s for s in sections if s.startswith("LOEData.csv")).split("\n") if l.startswith("Response")])
print(pd.crosstab(loe.Treatment, loe.Response).to_string())
print("Response equals (LOE > 0):", bool(((loe.LOE > 0).astype(int) == loe.Response).all()))
print("Spiral swimming in the R script:", [l.strip() for l in script.split("\n") if "Swirling<-" in l])
print("Columns of DepolarizationPeakandRecovery.csv:", list(depol.columns), "- Peak is the depolarization amplitude; no medulla peak amplitude")

CalciumPeaksLong.csv holds the experiment-2 medulla frequencies against time to the
depolarization, but no temperature and no onset temperature; the experiment-2 onsets of Fig. 2G
are not in the item (the only onsets are the 40 of experiment 5). The experiment-4 medulla counts
(Fig. S2B), the spiral-swimming and loss-of-response scores of Fig. S1B, C and E, the
light-response amplitudes of Fig. S4A and B, the medulla peak amplitudes and any imaging time
series are not deposited. The `Response` column of LOEData.csv is, by the Readme, whether a fish
ever lost equilibrium, and it equals LOE > 0 in every fish; it records nothing about loss of
response, so figs1e cannot be tested. The R script types the spiral-swimming counts into a matrix
(`matrix(c(0,14,10,2),nrow=2)`); they are not data in the item. The only `DepolFrame` column is a
frame index of onset. Nine claims are blocked for these reasons.

For figs1bc the criterion takes its counts from the caption of Table S2, and the R script types the
same counts in. The next cell runs the criterion's test on them. Since the counts are the paper's
and not deposited data, this checks the paper's arithmetic, not the dataset, and the claim stays
blocked.

In [ ]:
# figs1bc, the paper's arithmetic on the counts the criterion and the R script give (not data in the item)
swirling = np.array([[0, 10], [14, 2]])   # rows: spiral swimming yes, no; columns: control, heat ramp (matrix(c(0,14,10,2), nrow=2))
for correction in (True, False):
    chi2, p, dof, _ = stats.chi2_contingency(swirling, correction=correction)
    print(f"chi-squared {'with' if correction else 'without'} continuity correction: X2({dof}) = {chi2:.2f}, p = {p:.2g}")

The lines of the R script that the sections below follow are printed here, with their line
numbers, so that each can be checked against the script in the item. The script computes no Q10.

In [ ]:
cited = ["identify_outliers", "filter(CTmax>39)", "filter(Temperature>37)", 'p.val = "kr"', "filter(LowerTemp>27)",
         "stLowTemp<-", "CalciumPeaksLmm2<-", "filter(Temperature>28)", "Temperature<-LightResponse$Temperature-1",
         "centeredTemperature<-", "modelLightResponse1<-", "standTimeToEnd", "Frequency3min/3", "RecoveryMin<-",
         "Swirling<-", "wilcox.test", "lm(CTmax", "lm(Temperature", "lm(DepolTemp", "lm(Peak", "lm(RecoveryMin", "lmer("]
script_lines = script.split("\n")
for n, line in enumerate(script_lines, 1):
    if any(c.replace(" ", "") in line.replace(" ", "") for c in cited):
        print(f"{n:5d}: {line.strip()[:150]}")
print("\nlines mentioning Q10:", [n for n, l in enumerate(script_lines, 1) if re.search("q10|q 10", l, re.I)])

## fig1c-ctmax-5dpf

> Using loss of response as the end point, 5-day-old heat-ramp larvae reach CTmax at 41.4 ± 0.1 °C
> (mean ± SE, n = 12).

Criterion: mean over the 12 heat-ramp 5 dpf fish of experiment 1, tolerance 0.1 °C (the SE).

In [ ]:
five = ages[ages.AgeGroup == "5dpf"].CTmax
print(f"5 dpf: n = {five.size}, mean {five.mean():.3f} °C, SE {five.sem():.3f}, range {five.min()} to {five.max()}")
print("within 41.4 ± 0.1:", abs(five.mean() - 41.4) <= 0.1)

**Result: reproduced.** The 12 fish have a mean CTmax of 41.36 °C (SE 0.14), within 0.1 °C of 41.4.

## fig1c-ctmax-similar-across-ages

> CTmax is similar in 5 dpf larvae, 9 dpf larvae (41.3 ± 0.2 °C, n = 12) and adults tested by loss
> of equilibrium (41.0 ± 0.1 °C, n = 15), with no significant difference between the ages.

A claim of absence on the paper's test: Model 2 of Table S3, a linear model of CTmax on age
group after removing the extreme outlier (R script, section "Fig 1C": `identify_outliers`
within `AgeGroup`, then `filter(CTmax>39)`; `M2<-lm(CTmax~AgeGroup, ...)`). Model 1 (all 16 adults)
is a variant.

In [ ]:
ages["extreme"] = ages.groupby("AgeGroup").CTmax.transform(thermal.extreme_outliers)
print("Extreme outliers within age group:")
print(ages[ages.extreme].to_string())
reduced = ages[~ages.extreme]
print("\nMeans after removal:")
print(reduced.groupby("AgeGroup").CTmax.agg(["count", "mean", "sem"]).round(3).to_string())
print("Means of all fish (Model 1):")
print(ages.groupby("AgeGroup").CTmax.agg(["count", "mean", "sem"]).round(3).to_string())

def age_models(data, label):
    out = []
    for ref in ["5dpf", "9dpf"]:
        levels = [ref] + [a for a in ["5dpf", "9dpf", "Adult"] if a != ref]
        fit = smf.ols(f"CTmax ~ C(AgeGroup, levels={levels})", data).fit()
        for term in fit.params.index[1:]:
            out.append({"model": label, "reference": ref, "term": term.split("T.")[-1].rstrip("]"),
                        "estimate": fit.params[term], "se": fit.bse[term], "t": fit.tvalues[term], "p": fit.pvalues[term]})
    fit = smf.ols("CTmax ~ C(AgeGroup)", data).fit()
    print(f"{label}: n = {int(fit.nobs)}, F({int(fit.df_model)},{int(fit.df_resid)}) = {fit.fvalue:.2f}, p = {fit.f_pvalue:.3f}")
    return out

rows = age_models(reduced, "Model 2 (outlier removed)") + age_models(ages, "Model 1 (all fish)")
contrasts = pd.DataFrame(rows)
contrasts = contrasts[~((contrasts.reference == "9dpf") & (contrasts.term == "5dpf"))]
print(contrasts.round(3).to_string(index=False))
m2 = contrasts[contrasts.model.str.startswith("Model 2")]
print("\nModel 2, any pairwise p < 0.05:", bool((m2.p < 0.05).any()))

**Result: reproduced.** The rule removes one adult (38.9 °C). In Model 2 the 9 dpf mean is 41.26 °C
(within 41.3 ± 0.2) and the adult mean 40.95 °C (within 41.0 ± 0.1); the contrasts are 5 against 9
dpf p = 0.66, 5 dpf against adult p = 0.073 and 9 dpf against adult p = 0.18, and the overall
F(2,36) = 1.89, p = 0.17, as in Table S3 (0.7, 0.07, 0.2). With all 16 adults (Model 1) the adult contrast with 5 dpf
is significant (p = 0.038), so the absence rests on the outlier removal the paper made.

## figs1a-swim-speed-no-treatment-difference

> Swimming speed falls during the assay (from 2.9 ± 0.4 to 1.7 ± 0.4 mm/s), with no difference
> between control and heat-ramp larvae.

The R script, section "S1A Swimming speed": `Time` as a factor with five levels; the "Swim speed
model" `lmer(SwimSpeed ~ Time + (1|FishID), REML=T)` gives the expected values (intercept, and
intercept plus Time 5); the "Model including treatment" `lmer(SwimSpeed ~ Time + Treatment +
(1|FishID), REML=T)` gives the treatment test. The interaction model is a variant.

In [ ]:
swim["Time"] = swim.Time.astype(str)
speed = thermal.LMM("SwimSpeed ~ C(Time)", swim, "FishID")
speed_tab = speed.table()
print("Swim speed model:", speed.summary())
print(speed_tab.round(4).to_string())
with_treat = thermal.LMM("SwimSpeed ~ C(Time) + C(Treatment, levels=['Control', 'Heatramp'])", swim, "FishID")
treat_tab = with_treat.table()
print("\nModel including treatment:", with_treat.summary())
print(treat_tab.round(4).to_string())
interaction = thermal.LMM("SwimSpeed ~ C(Time) * C(Treatment, levels=['Control', 'Heatramp'])", swim, "FishID")
print("\nVariant with time by treatment interaction:")
print(interaction.table().round(4).to_string())
i1 = speed_tab.loc["Intercept", "estimate"]
i5 = i1 + speed_tab.loc["C(Time)[T.5]", "estimate"]
print(f"\nspeed at interval 1 {i1:.3f} mm/s, at interval 5 {i5:.3f} mm/s")
print("raw means by interval:", swim.groupby("Time").SwimSpeed.mean().round(3).to_dict())

**Result: reproduced.** The treatment effect is 0.76 mm/s (SE 0.62, t = 1.22, p = 0.23 with
Satterthwaite degrees of freedom), not significant, as in Table S1. The swim speed model gives
2.90 mm/s at interval 1 and 1.72 mm/s at interval 5, both within 0.4 of 2.9 and 1.7; the time 5
term is -1.18 mm/s (t(100) = -2.44, p = 0.017, the Table S1 value). No time by treatment
interaction term has p < 0.05 in the variant (smallest p 0.39), so the legend's "no difference in
the change" also holds.

## figs1d-loss-of-equilibrium

> Heat-ramp larvae spend more of the 12 min before CTmax in loss of equilibrium than controls do
> over the matching period.

The R script, section "S1D Loss of equilibrium": `wilcox.test(LOE ~ Treatment, exact=F)` (control
is the first level) and `rstatix::wilcox_effsize`, which computes r = |Z| / sqrt(N) from the
asymptotic coin statistic, without continuity correction.

In [ ]:
control = loe[loe.Treatment == "Control"].LOE.values
heat = loe[loe.Treatment == "Heatramp"].LOE.values
test = stats.mannwhitneyu(control, heat, alternative="two-sided", use_continuity=True, method="asymptotic")
ranks = stats.rankdata(np.concatenate([control, heat]))
n1, n2 = len(control), len(heat)
N = n1 + n2
_, tie_counts = np.unique(ranks, return_counts=True)
var_u = n1 * n2 / 12 * ((N + 1) - (tie_counts ** 3 - tie_counts).sum() / (N * (N - 1)))
z = (test.statistic - n1 * n2 / 2) / np.sqrt(var_u)
r = abs(z) / np.sqrt(N)
print(f"control n = {n1}, median {np.median(control):.2f} %; heat ramp n = {n2}, median {np.median(heat):.2f} %")
print(f"W (control first) = {test.statistic:.1f}, p = {test.pvalue:.4f} (continuity corrected), Z = {z:.3f}, r = {r:.3f}")
print("r within 0.58 ± 0.058:", abs(r - 0.58) <= 0.058)

**Result: reproduced.** Heat-ramp fish spent a median 15.3% of the period in loss of equilibrium
against 0.8% in controls; W = 27 (control first), p = 0.0035, r = 0.577 (within 0.58 ± 0.058), as
Table S2 reports.

## fig2f-medulla-control-stable

> In embedded control larvae held at 28 °C, medulla activity stays stable at 1.6 ± 0.1 calcium
> peaks/min, with no effect of time.

The R script, section "Fig 2F": `Frequency = round(Frequency3min/3, 3)` (per minute);
`standTimeToEnd = 42 - Interval` (minutes from the start of the measuring period);
`lmer(Frequency ~ Treatment*standTimeToEnd + Treatment*I(standTimeToEnd^2) + (1|FishID))` with
control as the reference, so the time and time squared main effects are those of the controls.
The bins are the deposited 3 min intervals. The 1.6 is read as (a) the mean over all control bins
and fish and (b) the mean of per-fish means; with 14 bins in every fish they are equal.

In [ ]:
peaks2["Frequency"] = (peaks2.Frequency3min / 3).round(3)
peaks2["t"] = 42 - peaks2.Interval
fig2f = thermal.LMM("Frequency ~ C(Treatment, levels=['Control', 'Heatramp']) * t + "
                    "C(Treatment, levels=['Control', 'Heatramp']) * I(t ** 2)", peaks2, "FishID")
fig2f_tab = fig2f.table()
print(fig2f.summary())
print(fig2f_tab.round(4).to_string())
ctrl = peaks2[peaks2.Treatment == "Control"]
print(f"\ncontrol mean over all bins and fish (a): {ctrl.Frequency.mean():.3f} peaks/min, SE over fish "
      f"{ctrl.groupby('FishID').Frequency.mean().sem():.3f}")
print(f"control mean of per-fish means (b): {ctrl.groupby('FishID').Frequency.mean().mean():.3f}")
print("bins per control fish:", ctrl.groupby("FishID").size().unique())
print("per control fish:", ctrl.groupby("FishID").Frequency.mean().round(2).to_dict())

**Result: reproduced.** Control fish average 1.58 peaks/min (within 1.6 ± 0.1; both readings are
the same, as every fish has 14 bins). The control time term is 0.018 per min (p = 0.39) and the
time squared term -0.0006 (p = 0.27), neither significant, matching Table S4 (p = 0.4 and 0.3).

## fig2f-medulla-heat-rise-then-fall

> In embedded heat-ramp larvae, medulla activity more than doubles with warming, to 3.6 ± 0.4
> peaks/min, then falls sharply below control levels before the brain-wide depolarization
> (significant time by heat ramp and time squared by heat ramp interactions).

The model of the previous section. The peak is read as (a) the maximum of the heat-ramp group's
mean curve over time bins and (b) the mean of each fish's maximum bin. The first bin is
Interval 42 (start of the measuring period) and the last before depolarization is Interval 3.

In [ ]:
hr = peaks2[peaks2.Treatment == "Heatramp"]
curve = hr.groupby("Interval").Frequency.agg(["mean", "sem", "count"]).sort_index(ascending=False)
print("Heat-ramp mean curve (Interval = minutes to depolarization):")
print(curve.round(3).T.to_string())
peak_a = curve["mean"].max()
peak_b = hr.groupby("FishID").Frequency.max().mean()
first = curve.loc[42, "mean"]
last = curve.loc[3, "mean"]
control_mean = ctrl.Frequency.mean()
print(f"\n(a) maximum of the mean curve {peak_a:.3f} at Interval {curve['mean'].idxmax()}; "
      f"(b) mean of per-fish maxima {peak_b:.3f} (SE {hr.groupby('FishID').Frequency.max().sem():.3f})")
print(f"first bin {first:.3f}; ratio (a)/first {peak_a / first:.2f}, (b)/first {peak_b / first:.2f}")
print(f"last bin before depolarization {last:.3f} against control mean {control_mean:.3f}")
inter = fig2f_tab.loc[[i for i in fig2f_tab.index if ":" in i]]
print(inter.round(5).to_string())
for reading, value in [("a", peak_a), ("b", peak_b)]:
    ok = (abs(value - 3.6) <= 0.4 and value >= 2 * first and last < control_mean
          and (inter.p < 0.001).all() and inter.estimate.iloc[0] > 0 and inter.estimate.iloc[1] < 0)
    print(f"criterion met under reading ({reading}):", ok)

**Result: reproduced under reading (a), not under (b).** The time by heat ramp term is 0.226
(t = 8.00) and time squared by heat ramp -0.0062 (t = -8.81), both p < 0.001, as in Table S4. The
heat-ramp mean curve peaks at 3.59 peaks/min 18 min before the depolarization (reading (a), within
3.6 ± 0.4), 2.08 times the first bin (1.73), and the last bin before the depolarization is 0.29,
below the control mean of 1.58. The mean of per-fish maxima (reading (b)) is 4.30, outside the
tolerance. The claims file lists both readings and the claim needs one, so the outcome is
reproduced.

## fig3d-ctmax-precedes-depolarization

> In freely swimming larvae imaged during the heat ramp, CTmax (40.9 ± 0.2 °C) precedes the
> brain-wide depolarization (41.4 ± 0.2 °C) in every fish, by 0.5 °C on average.

The R script, section "Fig 3D": `lmer(Temperature ~ Response + (1|FishID), REML=T)`, CTmax as the
reference.

In [ ]:
fig3 = thermal.LMM("Temperature ~ C(Response, levels=['CTmax', 'Depolarization'])", exp3, "FishID")
print(fig3.summary())
print(fig3.table().round(4).to_string())
paired = exp3.pivot(index="FishID", columns="Response", values="Temperature")
paired["difference"] = paired.Depolarization - paired.CTmax
print(paired.to_string())
print(f"\nmean CTmax {paired.CTmax.mean():.3f} (SE {paired.CTmax.sem():.3f}), mean depolarization "
      f"{paired.Depolarization.mean():.3f} (SE {paired.Depolarization.sem():.3f}), mean difference {paired.difference.mean():.3f}")
print("fish with depolarization above CTmax:", int((paired.difference > 0).sum()), "of", len(paired))

**Result: reproduced.** Mean CTmax 40.86 °C (SE 0.18) and mean onset 41.36 °C (SE 0.16), both within 0.2 of 40.9 and 41.4;
the difference is 0.50 °C (SE 0.11, t = 4.49, 6 Satterthwaite degrees of freedom, p = 0.004), and
the depolarization follows CTmax in all 7 fish (by 0.2 to 1.1 °C).

## fig4c-heart-rate-rises-to-34

> Heart rate of embedded larvae rises from 4.4 ± 0.1 Hz at 28 °C to 5.7 ± 0.1 Hz at 34 °C (Q10 1.5).

The R script, section "Fig 4C": `lmer(HeartBeat ~ Temperature + (1|FishID), REML=T)` with the
category as a factor and 28 °C as the reference. Means are of the fish with a value in the
category (the script's `HeartDataSummary`). Q10 as (R34 / R28) ^ (10 / 6) on group means; per-fish
Q10 as a variant.

In [ ]:
heart_levels = ["28", "31", "34", "37", "Depol", "After"]
def heart_model(ref):
    levels = [ref] + [l for l in heart_levels if l != ref]
    return thermal.LMM(f"HeartBeat ~ C(Temperature, levels={levels})", heart, "FishID")
h28 = heart_model("28")
h28_tab = h28.table()
print(h28.summary())
print(h28_tab.round(4).to_string())
hmeans = heart.groupby("Temperature").HeartBeat.agg(["count", "mean", "sem"]).loc[heart_levels]
print(hmeans.round(3).to_string())
q_28_34 = thermal.q10(hmeans.loc["28", "mean"], hmeans.loc["34", "mean"], 28, 34)
wide = heart.pivot(index="FishID", columns="Temperature", values="HeartBeat")
per_fish = ((wide["34"] / wide["28"]) ** (10 / 6)).dropna()
print(f"\nQ10 28 to 34 °C on group means {q_28_34:.3f}; mean of per-fish Q10 {per_fish.mean():.3f} (n = {per_fish.size})")

**Result: reproduced.** Heart rate is 4.45 Hz at 28 °C (8 fish) and 5.67 Hz at 34 °C (9 fish), both
within 0.1 Hz of the paper; the 34 against 28 °C term is 1.27 Hz (SE 0.18, t = 6.89, p < 0.001, as
Table S6), and Q10 is 1.50 on group means (1.5 ± 0.15; 1.55 as the mean of per-fish values).

## fig4c-heart-rate-falls-then-stable

> From 34 to 37 °C the heart rate falls (Q10 0.6) to 4.7 ± 0.2 Hz, then stays stable up to and
> through the brain-wide depolarization.

The same model releveled to 34 °C and to 37 °C (the paper's alpha for the releveled model is
0.001). Stability is judged on the depolarization against 37 °C contrast. Q10 as (R37 / R34) ^
(10 / 3) on group means.

In [ ]:
h34 = heart_model("34").table()
h37 = heart_model("37").table()
c_37_34 = h34.loc[[i for i in h34.index if i.endswith("[T.37]")]].iloc[0]
c_dep_37 = h37.loc[[i for i in h37.index if i.endswith("[T.Depol]")]].iloc[0]
print("37 against 34 °C:", c_37_34.round(4).to_dict())
print("depolarization against 37 °C:", c_dep_37.round(4).to_dict())
q_34_37 = thermal.q10(hmeans.loc["34", "mean"], hmeans.loc["37", "mean"], 34, 37)
per_fish = ((wide["37"] / wide["34"]) ** (10 / 3)).dropna()
print(f"mean at 37 °C {hmeans.loc['37', 'mean']:.3f} Hz (n = {int(hmeans.loc['37', 'count'])}); Q10 34 to 37 on group means {q_34_37:.3f}; "
      f"mean of per-fish Q10 {per_fish.mean():.3f}, median {per_fish.median():.3f}")
print("Q10 within 0.6 ± 0.06:", abs(q_34_37 - 0.6) <= 0.06, "; mean within 4.7 ± 0.2:", abs(hmeans.loc['37', 'mean'] - 4.7) <= 0.2)

**Result: reproduced.** Heart rate at 37 °C is 4.73 Hz (within 4.7 ± 0.2); the 37 against 34 °C
contrast is -0.91 Hz (t = -4.92, p < 0.001), and the depolarization category differs from 37 °C by
-0.11 Hz (p = 0.57), so the heart rate is stable through the depolarization by the paper's test.
The Q10 from 34 to 37 °C on group means is 0.550, inside 0.6 ± 0.06 by 0.01 (0.58 as the mean of
per-fish values).

## fig5a-ctmax-oxygen

> Oxygen level sets CTmax in freely swimming larvae. CTmax is 39.1 ± 0.1 °C in hypoxia (n = 24),
> 0.5 °C below normoxia (39.6 ± 0.1 °C, n = 28), and 40.1 ± 0.2 °C in hyperoxia (n = 21), 0.5 °C
> above normoxia.

The R script, section "Fig 5A": rows without a treatment dropped; `identify_outliers` within
treatment; `filter(Temperature>37)`; Model 2.2 `lm(Temperature ~ Treatment)` with normoxia
(`Control`) as reference. The claims file asks for outliers by quartiles within treatment and
over all fish; both are shown. Model 1 (all fish, with Experiment) is a variant.

In [ ]:
o2 = ct_o2.dropna(subset=["Treatment", "Temperature"]).copy()
o2["within"] = o2.groupby("Treatment").Temperature.transform(thermal.extreme_outliers)
o2["overall"] = thermal.extreme_outliers(o2.Temperature)
print("Extreme outliers within treatment:", o2[o2.within][["Treatment", "FishID", "Temperature"]].values.tolist())
print("Extreme outliers over all fish:", o2[o2.overall][["Treatment", "FishID", "Temperature"]].values.tolist())
print("Same fish removed by both rules:", bool((o2.within == o2.overall).all()))
lv = "C(Treatment, levels=['Control', 'Low', 'High'])"
for label, data, formula in [("Model 2.2, within-treatment outliers removed", o2[~o2.within], f"Temperature ~ {lv}"),
                             ("Model 2.2, overall outliers removed", o2[~o2.overall], f"Temperature ~ {lv}"),
                             ("Model 2.1, outliers removed, with Experiment", o2[~o2.within], f"Temperature ~ {lv} + C(Experiment)"),
                             ("Model 1, all fish, with Experiment", o2, f"Temperature ~ {lv} + C(Experiment)")]:
    fit = smf.ols(formula, data).fit()
    print(f"\n{label}: n = {int(fit.nobs)}, F({int(fit.df_model)},{int(fit.df_resid)}) = {fit.fvalue:.2f}, p = {fit.f_pvalue:.2g}")
    print(pd.DataFrame({"estimate": fit.params, "se": fit.bse, "t": fit.tvalues, "p": fit.pvalues}).round(4).to_string())
kept = o2[~o2.within]
print("\nMeans after removal:")
print(kept.groupby("Treatment").Temperature.agg(["count", "mean", "sem"]).round(3).to_string())
print("Means of all fish:")
print(o2.groupby("Treatment").Temperature.agg(["count", "mean", "sem"]).round(3).to_string())

**Result: reproduced.** The rule within treatment, which the R script applies
(`group_by(Treatment)`), removes three fish (33.8 and 34.8 °C in hyperoxia, 36.6 °C in hypoxia),
leaving 24, 28 and 21 fish, the paper's numbers. Mean CTmax is 39.07 °C in hypoxia, 39.65 °C in
normoxia and 40.14 °C in hyperoxia, each within the reported SE of the paper's means. In Model 2.2
hypoxia is 0.58 °C lower (t = -2.79, p = 0.007) and hyperoxia 0.49 °C higher (t = 2.29, p = 0.025)
than normoxia; F(2,70) = 11.67, as in Table S7. Quartiles over all fish remove only the two
hyperoxic fish; both contrasts stay significant (p = 0.003 and 0.035), but the hypoxic mean of 25
fish, 38.97 °C, is outside 39.1 ± 0.1. With all 76 fish (Model 1) neither contrast is significant,
so the outcome rests on the outlier removal the paper made.

## fig5b-depolarization-oxygen

> Oxygen level sets the depolarization onset in embedded larvae. It occurs at 40.2 ± 0.3 °C in
> normoxia (n = 12), 2.5 °C lower in hypoxia (37.7 ± 0.5 °C, n = 14) and 1.3 °C higher in hyperoxia
> (41.5 ± 0.2 °C, n = 14).

The R script, section "Fig 5B": `lm(DepolTemp ~ Treatment + Experiment)`, normoxia as reference.

In [ ]:
def o2_model(column, data=depol):
    fit = smf.ols(f"{column} ~ {lv} + C(Experiment)", data).fit()
    table = pd.DataFrame({"estimate": fit.params, "se": fit.bse, "t": fit.tvalues, "p": fit.pvalues})
    print(f"{column}: n = {int(fit.nobs)}, F({int(fit.df_model)},{int(fit.df_resid)}) = {fit.fvalue:.2f}, p = {fit.f_pvalue:.2g}")
    print(table.round(4).to_string())
    return table

onset_tab = o2_model("DepolTemp")
onset_means = depol.groupby("Treatment").DepolTemp.agg(["count", "mean", "sem"])
print(onset_means.round(3).to_string())
print(f"raw differences: hypoxia - normoxia {onset_means.loc['Low', 'mean'] - onset_means.loc['Control', 'mean']:.2f}, "
      f"hyperoxia - normoxia {onset_means.loc['High', 'mean'] - onset_means.loc['Control', 'mean']:.2f}")

**Result: reproduced.** Mean onset is 40.15 °C in normoxia, 37.71 °C in hypoxia and 41.49 °C in
hyperoxia, within the reported SEs. Adjusted for Experiment, hypoxia is 2.69 °C lower (t = -6.90,
p < 0.001) and hyperoxia 1.22 °C higher (t = 3.13, p = 0.003), with F(3,36) = 45.5, as in Table S8; the
raw differences are -2.44 and 1.34 °C.

## figs3b-amplitude-no-oxygen-effect

> Oxygen level does not change the amplitude of the brain-wide depolarization.

A claim of absence on the paper's test: `lm(Peak ~ Treatment + Experiment)` (R script, "S3B Peak
fluorescence in oxygen").

In [ ]:
peak_tab = o2_model("Peak")
print(depol.groupby("Treatment").Peak.agg(["count", "mean", "sem"]).round(2).to_string())

**Result: reproduced.** Neither contrast is significant: hypoxia 14.9 %ΔF/F0 above normoxia
(p = 0.32) and hyperoxia 10.6 below (p = 0.48), as in Table S8.

## figs3c-recovery-longer-in-hypoxia

> Hypoxia prolongs the recovery time after the depolarization compared with normoxia; hyperoxia does
> not shorten it significantly.

The R script, "S3C Recovery": `RecoveryMin = round(Recovery/60, 2)`;
`lm(RecoveryMin ~ Treatment + Experiment)`; the hyperoxic fish without a recovery time is dropped.

In [ ]:
depol["RecoveryMin"] = (depol.Recovery / 60).round(2)
rec_tab = o2_model("RecoveryMin")
print(depol.groupby("Treatment").RecoveryMin.agg(["count", "mean", "sem"]).round(3).to_string())

**Result: reproduced.** On 39 fish, hypoxia prolongs recovery by 2.57 min (SE 0.44, p < 0.001), the
Table S8 estimate, and hyperoxia shortens it by 0.71 min, not significantly (p = 0.12).

## figs3d-medulla-drops-before-depolarization

> In all three oxygen treatments, medulla activity drops sharply in the minutes before the brain-wide
> depolarization.

The claims file asks for frequencies aligned to the onset over the 30 min before it, and the group
mean over the last 2 min and the last 5 min before onset, judged against half the group's maximum
mean in the 30 min window. The item holds the experiment-5 frequencies only in 1 °C bins, with
each fish's onset temperature. With the ramp of 0.3 °C/min (Methods), a bin spans 3.3 min and its
time before onset is (onset - temperature) / 0.3. The window means are taken as the mean over the
bins weighted by their overlap with the window, per fish, then over fish; the 30 min profile uses
bins counted back from the onset bin (bin 0 holds the onset), as many as reach back 30 min (10 at
0.3 °C/min). This adaptation to the data's resolution is an assumption. The ramp is 0.3 °C/min in
the Methods and the Fig. 2 legend (experiment 5 uses the setup of experiment 2); the onset frames
of experiment 5 give a different slope across fish, so 0.225 °C/min is run as a variant, and the
start temperature each ramp implies is compared with the first medulla bin. The onset bin is cut
by the onset; neither the Readme nor the script says whether its frequency is over the part
recorded or over the whole bin, so the cell shows how it relates to that part, and a variant
rescales it as if it were over the whole bin. A reading outside the criterion, the last complete
bin before the onset bin, is shown as well. Fish without an onset (Fish33) are left out.

In [ ]:
RAMP = 0.3   # °C per minute (Fig. 2 legend, experiment 2; experiment 5 uses the same setup, Methods)
# Check of the ramp from the data: onset frame (5 Hz imaging, Methods) against onset temperature.
depol["onset_min"] = depol.DepolFrame / 5 / 60
for e, g in depol.groupby("Experiment"):
    slope, intercept = np.polyfit(g.onset_min, g.DepolTemp, 1)
    print(f"Experiment {e}: onset temperature = {intercept:.2f} + {slope:.3f} x minutes from the first frame, "
          f"r = {np.corrcoef(g.onset_min, g.DepolTemp)[0, 1]:.2f}, n = {len(g)}")
g = depol[(depol.Experiment == 1) & (depol.onset_min < 60)]
print(f"Experiment 1 without the {int(((depol.Experiment == 1) & (depol.onset_min >= 60)).sum())} recordings with onset after 60 min "
      f"({sorted(depol[(depol.Experiment == 1) & (depol.onset_min >= 60)].FishID)}): slope {np.polyfit(g.onset_min, g.DepolTemp, 1)[0]:.3f}, "
      f"r = {np.corrcoef(g.onset_min, g.DepolTemp)[0, 1]:.2f}, n = {len(g)}")
# Across fish the slope equals the ramp only if every recording starts at the same temperature. Each ramp
# implies a start temperature (onset temperature - ramp x onset time), to compare with the first 1 °C bin
# of the fish's medulla series.
starts = depol.join(peaks5.dropna(subset=["Frequency"]).groupby("FishID").LowerTemp.min().rename("first_medulla_bin"), on="FishID")
for ramp in (RAMP, 0.225):
    s = starts.DepolTemp - ramp * starts.onset_min
    print(f"ramp {ramp}: implied start temperature median {s.median():.2f} °C, quartiles {s.quantile(0.25):.2f} to "
          f"{s.quantile(0.75):.2f}, range {s.min():.2f} to {s.max():.2f}; by experiment {s.groupby(starts.Experiment).median().round(2).to_dict()}")
print("first medulla bin (lower bound, °C) of the fish with medulla values:", starts.first_medulla_bin.value_counts().to_dict())

base = peaks5[peaks5.LowerTemp > 27].dropna(subset=["Frequency"]).merge(
    depol[["FishID", "DepolTemp"]], on="FishID", how="inner")
base = base[base.LowerTemp <= np.floor(base.DepolTemp)].copy()
base["k"] = (np.floor(base.DepolTemp) - base.LowerTemp).astype(int)
print("fish:", base.groupby("Treatment").FishID.nunique().to_dict())

# The onset bin (k = 0) is cut by the onset. If its frequency had been divided by the whole bin's
# duration, it would scale with the part of the bin before the onset.
onset_bin = base[base.k == 0].copy()
onset_bin["fraction"] = onset_bin.DepolTemp - onset_bin.LowerTemp
onset_bin["previous_bin"] = onset_bin.FishID.map(base[base.k == 1].set_index("FishID").Frequency)
print("\nOnset bins:")
print(onset_bin[["FishID", "Treatment", "LowerTemp", "DepolTemp", "fraction", "Frequency", "previous_bin"]].round(2).to_string(index=False))
rho = stats.spearmanr(onset_bin.fraction, onset_bin.Frequency)
rho_rel = stats.spearmanr(onset_bin.fraction, onset_bin.Frequency / onset_bin.previous_bin, nan_policy="omit")
print(f"Spearman, fraction against onset-bin frequency: rho = {rho.statistic:.2f}, p = {rho.pvalue:.2f}; "
      f"against onset bin / previous bin: rho = {rho_rel.statistic:.2f}, p = {rho_rel.pvalue:.2f}; n = {len(onset_bin)}")
print("mean onset-bin frequency, fraction below 0.5:", round(onset_bin[onset_bin.fraction < 0.5].Frequency.mean(), 3),
      f"(n = {(onset_bin.fraction < 0.5).sum()}); 0.5 or more:", round(onset_bin[onset_bin.fraction >= 0.5].Frequency.mean(), 3),
      f"(n = {(onset_bin.fraction >= 0.5).sum()})")

def window_mean(d, minutes):
    overlap = (np.minimum(d.t_start, minutes) - np.maximum(d.t_end, 0)).clip(lower=0)
    return np.nan if overlap.sum() == 0 else float((d.Frequency * overlap).sum() / overlap.sum())

def drop_table(ramp, data=base):
    aligned = data.copy()
    aligned["t_end"] = (aligned.DepolTemp - np.minimum(aligned.LowerTemp + 1, aligned.DepolTemp)) / ramp   # minutes before onset
    aligned["t_start"] = (aligned.DepolTemp - aligned.LowerTemp) / ramp
    n_bins = int(np.ceil(30 * ramp)) + 1        # bins reaching back 30 min, the onset bin included
    rows = []
    for treatment, group in aligned.groupby("Treatment"):
        profile = group[group.k < n_bins].groupby("k").Frequency.mean()
        per_fish = {w: group.groupby("FishID").apply(window_mean, minutes=w).mean() for w in (2, 5)}
        last_full = group[group.k == 1].Frequency.mean()
        rows.append({"treatment": treatment, "fish": group.FishID.nunique(), "max_profile": profile.max(),
                     "k_of_max": int(profile.idxmax()), "last_2_min": per_fish[2], "last_5_min": per_fish[5],
                     "ratio_2": per_fish[2] / profile.max(), "ratio_5": per_fish[5] / profile.max(),
                     "ratio_last_full_bin": last_full / profile.max(),
                     "profile_from_onset_bin": profile.round(2).tolist()})
    return pd.DataFrame(rows)

for ramp in (RAMP, 0.225):
    s3d = drop_table(ramp)
    print(f"\nramp {ramp} °C/min" + (" (the paper's rate, decides the outcome)" if ramp == RAMP else " (variant, the slope of experiment 2 above)"))
    print(s3d.round(3).to_string(index=False))
    print("below half the maximum, last 2 min, every treatment:", bool((s3d.ratio_2 < 0.5).all()),
          "; last 5 min:", bool((s3d.ratio_5 < 0.5).all()))
    for share in (0.4, 0.6):
        print(f"threshold {share}: last 2 min {bool((s3d.ratio_2 < share).all())}, last 5 min {bool((s3d.ratio_5 < share).all())}")

# Variant: the onset-bin frequency taken as divided by the whole bin, so rescaled by the part before onset.
rescaled = base.copy()
k0 = rescaled.k == 0
rescaled.loc[k0, "Frequency"] = rescaled.loc[k0, "Frequency"] / (rescaled.loc[k0, "DepolTemp"] - rescaled.loc[k0, "LowerTemp"])
s3d_rescaled = drop_table(RAMP, rescaled)
print("\nvariant, onset bin rescaled by the part of the bin before onset (ramp 0.3):")
print(s3d_rescaled[["treatment", "max_profile", "k_of_max", "last_2_min", "last_5_min", "ratio_2", "ratio_5"]].round(3).to_string(index=False))

**Result: reproduced, resting on the adaptation to 1 °C bins.** Over the last 2 min before onset
the group means are 0.96 (normoxia), 0.93 (hyperoxia) and 0.87 peaks/min (hypoxia), 0.42, 0.29 and
0.49 of each group's maximum in the 30 min window, all below one half. Over the last 5 min the
ratios are 0.50, 0.36 and 0.64, so the 5 min reading fails in normoxia and hypoxia. The criterion
needs one reading, and the 2 min reading holds in every treatment, narrowly in hypoxia (0.49); with
a threshold of 0.4 in place of the claims file's 0.5 it would fail, and with 0.6 it holds.

The result rests on how the bins are placed in time, and three checks bear on it. With a ramp of
0.225 °C/min the outcome is the same (2 min ratios 0.42, 0.28 and 0.46). With 0.3 °C/min
the start temperature implied by the onset frames has a median of 27.0 °C (quartiles 26.1 to 27.5,
range 19.7 to 29.0), consistent with the first medulla bins (27 °C in 21 fish, 28 °C in 11), while
0.225 °C/min puts the start near 30 °C (median 29.95). This shows only that 0.3 °C/min is
compatible with the first bins. Across fish the onset
temperature rises 0.225 °C per minute of recording in the second period (r = 0.98) and 0.222 in the
first once Fish03, Fish04 and Fish05, whose onsets come after 60 min, are left out (r = 0.89; 0.086
and r = 0.56 with them), so the across-fish slope does not equal a constant 0.3 °C/min ramp
from 27 °C. The data cannot say which part of the ramp differs. Second, the onset-bin frequency hardly
depends on the part of the bin before the onset (Spearman rho = 0.23, p = 0.22, 30 fish; mean 0.84
peaks/min where that part is under half the bin, 0.92 where it is half or more), which is what
frequencies over the recorded part would give; if the frequency were over the whole bin and were
rescaled, the normoxic 2 min ratio would be 0.69 and the claim would fail. Third, the last complete
bin before the onset bin is 0.58, 0.40 and 0.72 of the maximum, so the drop below half is confined to
the partial onset bin in normoxia and hypoxia; in hyperoxia the last complete bin is already below
half. The outcome holds as the criterion defines it and as
the data are deposited, but it is fragile.

## fig5d-medulla-normoxia

> In normoxia, medulla activity rises with warming until 36 °C (Q10 1.8), to a maximum of 2.5 ±
> 0.4 peaks/min.

The R script, section "Fig 5D": bins from `LowerTemp` 28 on (`filter(LowerTemp>27)`), the bin
named by its lower bound (28 = 28 to 29 °C), group means per bin (`CalciumPeaksSummaryTreatments`,
"to find highest mean frequency"). Readings: (a) peak of the group mean curve, (b) mean of per-fish
maxima. Q10 on group means between 28 °C and the temperature of the maximum.

In [ ]:
p5 = peaks5[peaks5.LowerTemp > 27].copy()
curves = p5.dropna(subset=["Frequency"]).groupby(["Treatment", "LowerTemp"]).Frequency.agg(["mean", "sem", "count"])
print(curves.unstack(0).round(3).to_string())

def medulla_summary(treatment, t_high):
    c = curves.loc[treatment]
    fish_max = p5[p5.Treatment == treatment].dropna(subset=["Frequency"]).groupby("FishID").apply(
        lambda d: pd.Series({"max": d.Frequency.max(), "t_max": d.LowerTemp[d.Frequency.idxmax()]}))
    out = {"a_max": c["mean"].max(), "a_sem": c.loc[c["mean"].idxmax(), "sem"], "a_t_max": int(c["mean"].idxmax()),
           "b_max": fish_max["max"].mean(), "b_sem": fish_max["max"].sem(), "b_t_max": fish_max.t_max.mean(),
           "q10_28_to_t_max_a": thermal.q10(c.loc[28, "mean"], c["mean"].max(), 28, int(c["mean"].idxmax())),
           f"q10_28_to_{t_high}": thermal.q10(c.loc[28, "mean"], c.loc[t_high, "mean"], 28, t_high)}
    return pd.Series(out)

normoxia = medulla_summary("Control", 36)
print(normoxia.round(3).to_string())

**Result: reproduced under reading (a).** The normoxia mean curve peaks at 2.51 peaks/min (SE 0.35)
in the 36 °C bin, and Q10 from 28 to 36 °C is 1.71, all within tolerance (2.5 ± 0.4, 36 ± 1 °C,
1.8 ± 0.18). Under reading (b) the mean of per-fish maxima is 2.89 at a mean temperature of 34.9 °C,
just outside the one-bin tolerance on temperature.

## fig5d-medulla-hyperoxia

> In hyperoxia, medulla activity rises faster with warming (Q10 1.9) and reaches a higher maximum,
> 3.3 ± 0.3 peaks/min at 37 °C.

Binning and readings as for normoxia, Q10 from 28 to 37 °C. The model of Table S9 is the R
script's `CalciumPeaksLmm2`: `Frequency ~ Treatment*stLowTemp + I(stLowTemp^2) + I(stLowTemp^3) +
(1|FishID)`, with `stLowTemp = LowerTemp - 28` (°C from 28) and normoxia as reference.

In [ ]:
hyperoxia = medulla_summary("High", 37)
print(hyperoxia.round(3).to_string())
p5["st"] = p5.LowerTemp - 28
s9 = thermal.LMM("Frequency ~ C(Treatment, levels=['Control', 'Low', 'High']) * st + I(st ** 2) + I(st ** 3)", p5, "FishID")
s9_tab = s9.table()
print("\nTable S9 model:", s9.summary())
print(s9_tab.round(4).to_string())

**Result: reproduced.** The hyperoxia mean curve peaks at 3.23 peaks/min in the 37 °C bin (within
3.3 ± 0.3 and 37 ± 1 °C), and Q10 from 28 to 37 °C is 1.84 (within 1.9 ± 0.19). In the Table S9 model
(391 observations, 33 fish) the hyperoxia by temperature term is 0.089 per °C (t = 3.75, p = 0.0002),
as the paper reports.

## fig5d-medulla-hypoxia

> In hypoxia, medulla activity falls with temperature from the start of the ramp, at an
> accelerating rate (Q10 0.8 from 28 to 34 °C, 0.02 from 34 to 38 °C).

Binning and Q10 as above, with the text's endpoints; the second Q10 is judged as below 0.1
(claims file). The hypoxia by temperature term of the Table S9 model.

In [ ]:
c_low = curves.loc["Low"]
q_a = thermal.q10(c_low.loc[28, "mean"], c_low.loc[34, "mean"], 28, 34)
q_b = thermal.q10(c_low.loc[34, "mean"], c_low.loc[38, "mean"], 34, 38)
print(f"hypoxia means: 28 °C {c_low.loc[28, 'mean']:.3f} (n {int(c_low.loc[28, 'count'])}), 34 °C {c_low.loc[34, 'mean']:.3f} "
      f"(n {int(c_low.loc[34, 'count'])}), 38 °C {c_low.loc[38, 'mean']:.3f} (n {int(c_low.loc[38, 'count'])})")
print(f"Q10 28 to 34 °C {q_a:.3f} (0.8 ± 0.08: {abs(q_a - 0.8) <= 0.08}); Q10 34 to 38 °C {q_b:.4f} (below 0.1: {q_b < 0.1})")
term = [i for i in s9_tab.index if "Low" in i and ":st" in i][0]
print("hypoxia by temperature:", s9_tab.loc[term].round(5).to_dict())
# The 27 °C bin, which the R script leaves out (filter(LowerTemp>27), "before trial starts").
print("\n27 °C bin:", peaks5[peaks5.LowerTemp == 27].dropna(subset=["Frequency"]).groupby("Treatment").Frequency
      .agg(["count", "mean"]).round(3).to_dict("index"))
# Other readings of the endpoints. Bins named by their upper bound (as LightResponse.csv names them) make
# "28 to 34 °C" the bins 27 and 33; the bin midpoints (AvgTemp) name the same bins as the lower bounds.
low_all = peaks5[peaks5.Treatment == "Low"].dropna(subset=["Frequency"]).groupby("LowerTemp").Frequency.mean()
for (a, b), (c, d), label in [((28, 34), (34, 38), "lower-bound or midpoint labels (main)"),
                              ((27, 33), (33, 37), "upper-bound labels")]:
    print(f"{label}: Q10 28 to 34 °C {thermal.q10(low_all[a], low_all[b], 28, 34):.3f}, 34 to 38 °C {thermal.q10(low_all[c], low_all[d], 34, 38):.4f}")
# The Table S9 model's hypoxia curve (fixed effects) in place of the group means.
def s9_low(st):
    b = s9_tab.estimate
    low = "C(Treatment, levels=['Control', 'Low', 'High'])[T.Low]"
    return b["Intercept"] + b[low] + (b["st"] + b[low + ":st"]) * st + b["I(st ** 2)"] * st ** 2 + b["I(st ** 3)"] * st ** 3
print("Table S9 model, hypoxia:", {t: round(float(s9_low(t - 28)), 3) for t in (28, 34, 38)},
      f"Q10 28 to 34 °C {thermal.q10(s9_low(0), s9_low(6), 28, 34):.3f}, 34 to 38 °C {thermal.q10(s9_low(6), s9_low(10), 34, 38):.4f}")
# Per-fish Q10s: their mean is pulled up by fish with low rates at 28 °C, so the median is given too.
wide5 = p5[p5.Treatment == "Low"].pivot(index="FishID", columns="LowerTemp", values="Frequency")
both = wide5[[28, 34]].dropna()
per_fish_q = (both[34] / both[28]) ** (10 / 6)
print(f"variant, per-fish Q10 28 to 34 °C: mean {per_fish_q.mean():.3f}, median {per_fish_q.median():.3f}, "
      f"range {per_fish_q.min():.2f} to {per_fish_q.max():.2f} (n = {len(both)})")
both38 = wide5[[34, 38]].dropna()
print(f"variant, Q10 34 to 38 °C on the {len(both38)} fish with both bins: {thermal.q10(both38[34].mean(), both38[38].mean(), 34, 38):.4f}")

**Result: partially reproduced.** Hypoxic activity falls with temperature at an accelerating rate:
the Q10 from 34 to 38 °C is 0.027 (below 0.1) and the hypoxia by temperature term is -0.161 per °C
(t = -5.67, p < 0.001). The Q10 from 28 to 34 °C is 0.88, just outside 0.8 ± 0.08 (the bound is 0.88;
the unrounded value 0.883 exceeds it), so one of the three parts of the criterion is not met. The R
script computes no Q10 (no line mentions it), so it cannot settle the endpoints. No other reading
shown gives 0.8: with bins named by their upper bound the first Q10 is 1.38 (the second 0.12); on
the Table S9 model's hypoxia curve it is 0.63 (the second 0.088); the median of per-fish values is
0.71, and their mean, 2.03, is pulled up by fish with low rates at 28 °C (per-fish values 0.27 to
14.4). The 27 °C bin, which the script leaves out as before the trial starts, holds only 3 hypoxic
fish (mean 1.44).

## fig6c-light-responsiveness-by-oxygen

> Optic tectum responsiveness to light declines with heating, and its resilience depends on oxygen.
> In normoxia about 31% of flashes evoke a response until 31 °C, then fewer; in hypoxia responses
> decline from 33% from the start; in hyperoxia 36% is kept up to 36 °C, a gain in resilience of
> 5 °C.

The deposited binary outcomes are used (counts of responses per fish and 1 °C bin). The R script
("Fig 6D Light response", the code of Fig. 6C) drops the first bin (`filter(Temperature>28)`),
labels bins by their lower bound (`Temperature - 1`), and plots the mean of per-fish percentages;
the claims file's main reading pools flashes over fish, and both are shown. Initial level: mean
over bins 28 to 31 °C (normoxia), 28 to 36 °C (hyperoxia), and the first bin (hypoxia).
Temperature of decline: the last bin before the percentage falls below the initial level minus
its tolerance. As a further variant, bins named by their upper bound.

In [ ]:
lr = light[light.Temperature > 28].copy()
lr["lower"] = lr.Temperature - 1
lr["n"] = lr["number.of.successes"] + lr["number.of.failures"]
pooled = lr.groupby(["Treatment", "lower"]).apply(lambda d: 100 * d["number.of.successes"].sum() / d.n.sum())
per_fish = lr.dropna(subset=["prob"]).groupby(["Treatment", "lower"]).prob.mean() * 100
fish_n = lr.dropna(subset=["prob"]).groupby(["Treatment", "lower"]).FishID.nunique()
print(pd.concat({"pooled": pooled.unstack(0), "per_fish": per_fish.unstack(0), "fish": fish_n.unstack(0)}, axis=1).round(1).to_string())

def judge(curve, label_shift=0):
    def series(treatment):
        c = curve.loc[treatment].copy()
        c.index = c.index + label_shift
        return c
    out = {}
    for treatment, top, expected, tol in [("Control", 31, 31, 3.1), ("High", 36, 36, 3.6)]:
        c = series(treatment)
        top = top + label_shift
        initial = c.loc[28 + label_shift: top].mean()
        below = c[(c.index > 28 + label_shift) & (c < initial - tol)]
        decline = (below.index.min() - 1) if len(below) else np.nan
        out[treatment] = {"initial": initial, "initial_ok": abs(initial - expected) <= tol,
                          "kept_to": decline, "kept_ok": abs(decline - expected) <= 1}
    c = series("Low")
    out["Low"] = {"initial": c.iloc[0], "initial_ok": abs(c.iloc[0] - 33) <= 3.3,
                  "falls": bool(c.iloc[1] < c.iloc[0])}
    return pd.DataFrame(out).T

for name, curve, shift in [("pooled, lower-bound labels", pooled, 0), ("per-fish mean, lower-bound labels", per_fish, 0),
                           ("pooled, upper-bound labels", pooled, 1), ("per-fish mean, upper-bound labels", per_fish, 1)]:
    print(f"\n{name}:")
    print(judge(curve, shift).to_string())

# Variant, not part of the criterion: the last bin after which the hyperoxic percentage stays below the threshold.
c = pooled.loc["High"]
init = c.loc[28:36].mean()
stays = [t for t in c.index if (c.loc[t:] < init - 3.6).all()]
print(f"\nhyperoxia, pooled: first bin from which the percentage stays below {init - 3.6:.1f}%: {min(stays)} °C")

**Result: partially reproduced.** The percentages match under both readings: normoxia 33.0%
(pooled) or 31.7% (per fish) over 28 to 31 °C, hyperoxia 36.5% or 36.2% over 28 to 36 °C, hypoxia
36.1% or 33.4% in the first bin, falling to about 20% in the next. Normoxic responsiveness is kept
to 31 °C (it falls below the threshold at 32 °C). In hyperoxia the 30 °C bin (31.6% pooled, 28.2%
per fish) already dips below the initial level minus its tolerance, so by the criterion's rule the
level is kept only to 29 °C, not 36 °C; from 37 °C on the percentage stays below the threshold. With
upper-bound labels the hyperoxic temperature is 30 °C. The hyperoxic temperature, and so the 5 °C
gain, is not met as the criterion defines it.

## tables10-light-response-model

> In a binomial mixed model, responsiveness at mean temperature is lower in hypoxia (13%) and higher
> in hyperoxia (37%) than in normoxia (26%), falls with temperature, and hyperoxia slows its decline
> with temperature.

The R script, "Fig 6D Light response": `modelLightResponse1 <- glmer(cbind(successes, failures) ~
Treatment*centeredTemperature + Treatment*I(centeredTemperature^2) + (1|FishID), family =
binomial)` on rows with `Temperature > 28`, temperature centered on the mean over those rows
(`scale(..., scale = F)`, all treatments; rows without counts included in the mean). The claims
file asks also for centering on the mean over flashes and per treatment; those are shown by
predicting from the same model at the other means. Laplace approximation (glmer's default) and,
as a check, 9-node adaptive quadrature.

In [ ]:
lr = light[light.Temperature > 28].copy()
centre_rows = lr.Temperature.mean()
lr["cT"] = lr.Temperature - centre_rows
formula = "C(Treatment, levels=['Control', 'Low', 'High']) * cT + C(Treatment, levels=['Control', 'Low', 'High']) * I(cT ** 2)"
glmm = thermal.BinomialGLMM(formula, lr, "number.of.successes", "number.of.failures", "FishID", nagq=1)
s10 = glmm.table()
s10.index = [i.replace("C(Treatment, levels=['Control', 'Low', 'High'])", "Treatment") for i in s10.index]
print(f"Laplace fit: {len(glmm.k)} rows, {int(glmm.m.sum())} flashes, {glmm.G} fish, fish SD {glmm.sd_fish:.3f}, "
      f"log-likelihood {glmm.loglik:.2f}, converged {glmm.converged}")
print(s10.round(4).to_string())
check = thermal.BinomialGLMM(formula, lr, "number.of.successes", "number.of.failures", "FishID", nagq=9).table()
print("largest change in an estimate with 9 quadrature nodes:", float(np.abs(check.estimate.values - s10.estimate.values).max().round(5)))

b = s10.estimate
def prob(treatment, at):
    x = at - centre_rows
    eta = b["Intercept"] + b["cT"] * x + b["I(cT ** 2)"] * x ** 2
    if treatment != "Control":
        t = "Treatment[T.%s]" % treatment
        eta += b[t] + b[t + ":cT"] * x + b[t + ":I(cT ** 2)"] * x ** 2
    return 100 * thermal.inverse_logit(eta)

centre_flashes = np.average(lr.dropna(subset=["prob"]).Temperature, weights=lr.dropna(subset=["prob"]).eval("`number.of.successes` + `number.of.failures`"))
centre_treat = lr.groupby("Treatment").Temperature.mean()
print(f"\nmean temperature over rows {centre_rows:.3f}, over flashes {centre_flashes:.3f}; per treatment {centre_treat.round(3).to_dict()}")
probs = pd.DataFrame({"over rows (script)": {t: prob(t, centre_rows) for t in ["Control", "High", "Low"]},
                      "over flashes": {t: prob(t, centre_flashes) for t in ["Control", "High", "Low"]},
                      "per treatment": {t: prob(t, centre_treat[t]) for t in ["Control", "High", "Low"]}})
print(probs.round(2).to_string())
expected = {"Control": (26, 2.6), "High": (37, 3.7), "Low": (13, 1.3)}
for col in probs:
    print(col, "all within tolerance:", all(abs(probs.loc[t, col] - v) <= tol for t, (v, tol) in expected.items()))

**Result: reproduced.** On 411 rows with counts (2763 flashes, 38 fish) the hypoxia term is -0.85 (SE 0.22,
p < 0.001), the hyperoxia term 0.49 (SE 0.18, p = 0.008), temperature -0.13 per °C (p < 0.001) and
hyperoxia by temperature 0.065 (p = 0.030), as in Table S10. At the script's mean temperature
(34.5 °C) the probabilities are 26.2% (normoxia), 36.6% (hyperoxia) and 13.2% (hypoxia), within
tolerance; centering over flashes gives 26.1%, 36.5% and 13.2%, also within tolerance, while
predicting at each treatment's own mean temperature puts hypoxia at 14.5%, outside 13 ± 1.3. The
script's centering matches.

## Summary

| Claim | Outcome |
|---|---|
| fig1c-ctmax-5dpf | reproduced |
| fig1c-ctmax-similar-across-ages | reproduced |
| figs1a-swim-speed-no-treatment-difference | reproduced |
| figs1bc-spiral-swimming | blocked, data missing |
| figs1d-loss-of-equilibrium | reproduced |
| figs1e-loss-of-response-specific | blocked, data missing |
| fig2f-medulla-control-stable | reproduced |
| fig2f-medulla-heat-rise-then-fall | reproduced |
| figs2a-medulla-decrease-39-41 | blocked, data missing |
| fig2g-depolarization-onset-temperature | blocked, data missing |
| fig2e-depolarization-spread | blocked, data missing |
| fig2d-recovery-after-depolarization | blocked, data missing |
| fig3d-ctmax-precedes-depolarization | reproduced |
| fig4c-heart-rate-rises-to-34 | reproduced |
| fig4c-heart-rate-falls-then-stable | reproduced |
| figs2b-medulla-lateral-mount | blocked, data missing |
| fig5a-ctmax-oxygen | reproduced |
| fig5b-depolarization-oxygen | reproduced |
| figs3b-amplitude-no-oxygen-effect | reproduced |
| figs3c-recovery-longer-in-hypoxia | reproduced |
| figs3d-medulla-drops-before-depolarization | reproduced (assumed) |
| fig5d-medulla-normoxia | reproduced |
| fig5d-medulla-hyperoxia | reproduced |
| fig5d-medulla-hypoxia | partially reproduced |
| figs4ab-light-response-amplitude-falls | blocked, data missing |
| fig6c-light-responsiveness-by-oxygen | partially reproduced |
| tables10-light-response-model | reproduced |
| discussion-depolarization-magnitude | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's: it runs in Python, not R; mixed-model p-values use
Satterthwaite degrees of freedom where the R script's tables use Kenward-Roger; the medulla drop
before the depolarization is judged on 1 °C bins converted to time with the 0.3 °C/min ramp,
since the item holds no time-resolved experiment-5 frequencies; in normoxia and hypoxia that result rests
on the partial bin that holds the onset, and it is fragile.

What was found about the dataset: every table matches the paper's counts, and every
estimate of Tables S1 and S3 to S10 that the claims file quotes is reproduced; the notebook does
not compare the other terms of those tables. The item does not hold the experiment-2
temperatures or onset temperatures, the experiment-4 medulla counts, the spiral-swimming and
loss-of-response scores, the light-response amplitudes, the medulla peak amplitudes or any
imaging time series, so nine claims cannot be tested. The medulla series of two fish end 1 to 2 °C below the bin of their
onset (Fish03, Fish35), one hypoxic fish (Fish33) has medulla and light values but no
depolarization row, and three FishIDs of LightResponse.csv are spelled without the zero padding
of the other tables. The Readme gives the recovery time in minutes, but the values are seconds.

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s")